In [110]:
# Import dependencies

import os
import wandb
import wandb_workspaces.workspaces as ws
import wandb_workspaces.reports.v2 as wr # We use the Reports API for adding panels

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [111]:
WANDB_API_KEY = "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"
ENTITY = "kirill456z"
PROJECT = "physics4llm"

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
if ENTITY:
    os.environ["WANDB_ENTITY"] = ENTITY
if PROJECT:
    os.environ["WANDB_PROJECT"] = PROJECT


In [112]:
#import wandb_workspaces.reports.v2 as wr

#report = wr.Report(
     #entity=ENTITY,
     #project=PROJECT,
     #title="20.04",
     #description="Canon final exps",
#)

#report.save();

In [113]:
report = wr.Report.from_url("https://wandb.ai/kirill456z/physics4llm/reports/20-04--VmlldzoxNjU1MDUwMA")

report.width = 'fluid'

In [114]:

from plots import canon_weight_fixed_layer, speed_cur_iter_time
from plots import loss_lineplot, to_matplotlib
from runsets import RunsetsFactory

runsets_factory = RunsetsFactory(ENTITY, PROJECT)
FAST_PLOT_KWARGS = dict(api_timeout=90, max_points_per_series=150, use_scan_fallback=False)


seed_variance_2_comp = [
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_768_0.1.280",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_768_0.1.281"
]

seed_variance_2_comp_runs = [runsets_factory.filter_by_run_names(seed_variance_2_comp)]
loss_plots = [loss_lineplot(title="Baseline LLama, varying data & model seeds (5 tasks / edges_list enc)", y_max=3, smoothing_factor=0.0)]

report.blocks = [
    wr.H2("Mix of tasks 20/20/20/20/20 edges_list"),
    wr.PanelGrid(
        runsets = seed_variance_2_comp_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
]


seed_variance_comp = [
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.1.303",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.1.304",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.1.305"
]



seed_var_comp_runs = [runsets_factory.filter_by_run_names(seed_variance_comp)]
loss_plots = [loss_lineplot(title="Baseline LLama, varying data & model seeds (5 tasks / edges_list & adjacency_list enc)", y_max=3, smoothing_factor=0.0)]

report.blocks += [
    wr.H2("Mix of tasks 20/20/20/20/20 edges_list adjacency_list"),
    wr.P("Varying data and model init seed, LLama"),
    wr.PanelGrid(
        runsets = seed_var_comp_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
]


In [115]:
from utils import batch_rename_wandb_run

renames = {
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.0": "canon_lr_1e-4",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.1": "canon_lr_2e-4",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.2": "canon_lr_4e-4",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.10": "canon_lr_6e-4",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.5": "canon_lr_1e-3",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.6": "llama_lr_2e-4",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.7": "llama_lr_4e-4",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.9": "llama_lr_6e-4",
}

#res = batch_rename_wandb_run(renames)
#print(res)

lr_rate_swipes = list(renames.keys())

lr_rate_swipes_runs = [runsets_factory.filter_by_run_names(lr_rate_swipes)]
loss_plots = [loss_lineplot(title="LLama & Canon, varying lr rates", y_max=3, smoothing_factor=0.0)]

report.blocks += [
    wr.H2("LR rates optimization"),
    wr.PanelGrid(
        runsets = lr_rate_swipes_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
    wr.P("6e-4 seems to be optimal, but now Canon doesn't outperform LLama")
]

In [116]:
renames = {
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.20": "canon_data_mix",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.18": "llama_data_mix",
}

#res = batch_rename_wandb_run(renames)

complete_comp = list(renames.keys())

complete_comp_runs = [runsets_factory.filter_by_run_names(complete_comp)]

loss_plots= [
    loss_lineplot(title="Final loss", y_max=3, smoothing_factor=0.0)
]

report.blocks += [
    wr.H2("Loss breakdown"),
    wr.PanelGrid(
        runsets = complete_comp_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
]

tasks = ['depo', 'brevo', 'concomp', 'concomp_factor', 'sp']
encoding = ['edges_list', 'adjacency_list']

loss_plots = []
for task in tasks:
    for enc in encoding:
        loss_plots.append(loss_lineplot(title=f"{task} ({enc})", y_max=3, smoothing_factor=0.0, name = f"{task}/{enc}"))

from plots import realign_grid
loss_plots = realign_grid(loss_plots, num_columns = 2, total_w = 25, plot_h = 7)

report.blocks += [
    wr.PanelGrid(
        runsets = complete_comp_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
]

In [117]:
renames = {
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.28": "canon_data_mix_edges_list_only",
    "mix_depo20_brevo20_concomp20_concomp_factor20_sp20_bs_256_seq_len_1024_0.2.27": "llama_data_mix_edges_list_onlly"
}

res = batch_rename_wandb_run(renames)

complete_comp = list(renames.keys())

complete_comp_runs = [runsets_factory.filter_by_run_names(complete_comp)]

loss_plots= [
    loss_lineplot(title="Final loss", y_max=3, smoothing_factor=0.0)
]

report.blocks += [
    wr.H2("Loss breakdown : Only edges list"),
    wr.PanelGrid(
        runsets = complete_comp_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
]

tasks = ['depo', 'brevo', 'concomp', 'concomp_factor', 'sp']
encoding = ['edges_list']

loss_plots = []
for task in tasks:
    for enc in encoding:
        loss_plots.append(loss_lineplot(title=f"{task} ({enc})", y_max=3, smoothing_factor=0.0, name = f"{task}/{enc}"))

from plots import realign_grid
loss_plots = realign_grid(loss_plots, num_columns = 1, total_w = 25, plot_h = 7)

report.blocks += [
    wr.PanelGrid(
        runsets = complete_comp_runs,
        panels = loss_plots,
        hide_run_sets = True
    ),
]

In [118]:
report.save();

wandb: Saved report to: https://wandb.ai/kirill456z/physics4llm/reports/20.04--VmlldzoxNjU1MDUwMA==
